# Appendix: Chronos-2 vs TiRex-2 — context windows & data gaps

Two questions the core notebook glosses over:

1. **How much history does each model actually use?** Chronos-2 has a fixed
   token context window (~8192 tokens; with 1 target + 2 covariates per day that
   is ~7.5 years of daily values). TiRex-2 has its own context length.
2. **What happens when the target history has gaps?** Real observation records
   have missing days. Both models are trained to forecast from incomplete
   context, but the effect on skill is worth seeing. Here we take a clean series,
   punch synthetic gaps into it, and compare the forecasts.

This is a diagnostic notebook — it deliberately runs on a single site so it stays
fast in a live session.


In [ ]:
# CPU torch first so tirex-2's flashrnn CUDA kernels don't try (and fail) to
# build on Colab's default GPU; then numpy pinned so later installs don't leave a
# half-upgraded numpy. If a numpy ImportError appears on Colab, restart & re-run.
!pip install -q "torch<2.10" torchvision --index-url https://download.pytorch.org/whl/cpu
!pip install -q "numpy>=1.26,<2.1" 'chronos-forecasting[extras]>=2.2' tirex-2 pandas matplotlib

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from chronos import BaseChronosPipeline

# A synthetic but realistic daily series: seasonal cycle + AR(1) noise + spikes.
n = 365 * 4
rng = np.random.default_rng(0)
t = np.arange(n)
seasonal = 20 + 15 * np.sin(2 * np.pi * t / 365)
noise = np.zeros(n)
for i in range(1, n):
    noise[i] = 0.8 * noise[i - 1] + rng.normal(0, 3)
series = np.clip(seasonal + noise, 1, None)
idx = pd.date_range("2022-01-01", periods=n, freq="D")
clean = pd.Series(series, index=idx, name="value")

HORIZON = 15
train = clean.iloc[:-HORIZON]
truth = clean.iloc[-HORIZON:]


## Report each model's context length

In [ ]:
chronos = BaseChronosPipeline.from_pretrained("amazon/chronos-2", device_map="auto")
print("Chronos-2 context length (tokens):", chronos.model_context_length)

from tirex2 import load_model
device = "cuda" if torch.cuda.is_available() else "cpu"
tirex = load_model("NX-AI/TiRex-2", device=device)
print("TiRex-2 loaded on", device)


## Forecast helpers (target-only, no covariates)

To isolate the gap effect we forecast the univariate target with no weather
covariates. Both APIs accept a bare context series.


In [ ]:
from tirex2 import TimeseriesType

_Q = [round(float(q), 3) for q in tirex.quantiles.detach().cpu().numpy()]
_MEDIAN_Q = _Q.index(0.5)

def chronos_forecast(context_series):
    df = context_series.rename("value").reset_index().rename(columns={"index": "timestamp"})
    df.columns = ["timestamp", "value"]
    df["id"] = "s"
    pred = chronos.predict_df(
        df, prediction_length=HORIZON, quantile_levels=[0.5],
        id_column="id", timestamp_column="timestamp", target="value",
    )
    return pred["predictions"].to_numpy()

def tirex_forecast(context_series):
    target = context_series.to_numpy(dtype="float32")[None, :]  # (1, context_len), NaNs allowed
    ts = TimeseriesType(target=torch.from_numpy(target), past_covariates=None,
                        future_covariates=None)
    fc = tirex.forecast(timeseries=[ts], prediction_length=HORIZON, output_type="numpy")[0]
    return np.asarray(fc)[0, _MEDIAN_Q, :]  # median quantile


## Punch synthetic gaps

We drop random days from the *recent* history (last year) at increasing rates and
re-forecast. Chronos-2's `predict_df` needs a contiguous frame, so we forward-fill
its input as a simple imputation; TiRex-2 accepts NaNs directly.


In [ ]:
def with_gaps(series, frac, seed):
    r = np.random.default_rng(seed)
    s = series.copy()
    recent = s.index[-365:]
    drop = r.choice(recent, size=int(frac * len(recent)), replace=False)
    s.loc[drop] = np.nan
    return s

rmse = lambda a, b: float(np.sqrt(np.nanmean((np.asarray(a) - np.asarray(b)) ** 2)))

records = []
for frac in [0.0, 0.1, 0.3, 0.5]:
    gapped = with_gaps(train, frac, seed=1)
    c_pred = chronos_forecast(gapped.ffill())      # Chronos: impute gaps
    t_pred = tirex_forecast(gapped)                # TiRex-2: native NaN
    records.append({"gap_frac": frac,
                    "Chronos-2 RMSE": rmse(c_pred, truth.values),
                    "TiRex-2 RMSE": rmse(t_pred, truth.values)})
skill = pd.DataFrame(records).set_index("gap_frac")
skill


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
skill.plot(marker="o", ax=ax)
ax.set_xlabel("Fraction of last-year history missing")
ax.set_ylabel(f"{HORIZON}-day forecast RMSE")
ax.set_title("Forecast skill vs. gaps in the target history")


## Takeaways

- Both models degrade gracefully as gaps grow, rather than failing.
- TiRex-2 ingests NaNs natively; for Chronos-2 you must impute (here a simple
  forward-fill) before `predict_df`. How you impute matters — a spike-preserving
  method may beat forward-fill for flashy hydrographs.
- In the core notebook we keep **target** gaps and only require **covariates** to
  be present, which mirrors the more robust TiRex-2 path. If you rely on
  Chronos-2 with a very gappy record, consider a better imputation than the
  default.
